[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarcusBae/EYE-D/blob/feat/phase1/edge/notebooks/run_pipeline.ipynb)

# Re-ID 파이프라인 실행 노트북

이 노트북은 `run_pipeline.sh`를 Google Colab에서 실행하여 영상에서 Re-ID 임베딩을 추출합니다.

```
data/*.avi  →  Stage 1 (트랙 수집)  →  Stage 2 (Re-ID 추출)  →  results/*.pkl
```

## 전제 조건

- **Google Drive** 에 아래 구조로 파일이 준비되어 있어야 합니다:

```
MyDrive/projects/EYE-D/EYE-D/
├── data/        ← 영상 파일 (*.avi)
├── edge/        ← 소스 코드 (이 노트북 포함)
├── tracks/      ← Stage 1 결과 자동 생성
├── results/     ← Stage 2 결과 자동 생성
└── dataset/     ← 학습용 크롭 이미지 자동 생성 (IMAGE_DIR 지정 시)
```

- **런타임**: GPU (T4 이상 권장) — *런타임 → 런타임 유형 변경 → T4 GPU*
- 셀 1 → 5를 **순서대로** 실행하세요.

> **재실행 안전**: 두 Stage 모두 1000 프레임마다 체크포인트를 저장합니다.  
> 중단되어도 같은 명령으로 재실행하면 이어서 처리합니다.

## 목차

| # | 섹션 | 내용 |
|---|------|------|
| 1 | [Google Drive 마운트](#1.-Google-Drive-마운트) | Drive 연결 |
| 2 | [최신 코드 가져오기](#2.-최신-코드-가져오기) | git pull로 최신 코드 반영 |
| 3 | [파라미터 설정](#3.-파라미터-설정) | 경로·실행 옵션 설정 |
| 4 | [패키지 설치](#4.-패키지-설치) | 필요 라이브러리 설치 |
| 5 | [파이프라인 실행](#5.-파이프라인-실행) | Stage 1 → Stage 2 순차 실행 |
| 6 | [결과 확인](#6.-결과-확인) | 생성된 pkl 내용 출력 |

### 1. Google Drive 마운트


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


### 2. 최신 코드 가져오기


In [ ]:
import subprocess, os, shutil

REPO_URL    = 'https://github.com/MarcusBae/EYE-D.git'
REPO_BRANCH = 'feat/phase1'                                  # ← 브랜치 확인
REPO_LOCAL  = '/content/drive/MyDrive/projects/EYE-D/EYE-D' # ← DRIVE_ROOT와 동일하게
TMP_CLONE   = '/content/_ey_d_clone'                         # 로컬 임시 클론 경로

# Drive FUSE 위에서 git 실행 시 credential 오류가 발생하므로
# 로컬 /content/ 에 clone 후 코드 파일만 Drive로 복사
print('최신 코드 다운로드 중...')
if os.path.exists(TMP_CLONE):
    shutil.rmtree(TMP_CLONE)

r = subprocess.run(
    ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, TMP_CLONE],
    text=True, capture_output=True
)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise RuntimeError(f'git clone 오류: {r.stderr}')

# 코드 파일만 Drive로 복사 (영상·결과 데이터 폴더 제외)
EXCLUDE = {'.git', 'data', 'results', 'tracks', 'dataset'}
os.makedirs(REPO_LOCAL, exist_ok=True)
for item in os.listdir(TMP_CLONE):
    if item in EXCLUDE:
        continue
    src = os.path.join(TMP_CLONE, item)
    dst = os.path.join(REPO_LOCAL, item)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

shutil.rmtree(TMP_CLONE)
print('완료')


### 3. 파라미터 설정


In [ ]:
# ── 경로 설정 ─────────────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/projects/EYE-D/EYE-D'   # ← 필요 시 수정
DATA_DIR   = f'{DRIVE_ROOT}/data'
SCRIPT     = f'{DRIVE_ROOT}/edge/notebooks/run_pipeline.sh'

# ── 처리할 영상 ────────────────────────────────────────────────────────────────
import glob, os
VIDEO_FILES = sorted(glob.glob(f'{DATA_DIR}/*.avi'))
print(f'처리 대상 영상 {len(VIDEO_FILES)}개:')
for v in VIDEO_FILES:
    print(f'  {os.path.basename(v)}')


In [ ]:
# ── 실행 옵션 ─────────────────────────────────────────────────────────────────
IMAGE_DIR   = 'dataset'   # 학습용 이미지 저장 폴더 ('' = 저장 안 함)
FRAME_STEP  = 5           # Stage 2: N번째 프레임만 ReID 추출
MAX_FRAMES  = 'inf'       # 최대 처리 프레임 (inf = 전체)
CLEAN       = '0'         # '1' = 기존 결과 삭제 후 재실행


### 4. 패키지 설치


In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'papermill', 'ultralytics', 'boxmot'], check=True)
# deep-person-reid (OSNet)
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/KaiyangZhou/deep-person-reid.git'], check=True)
print('설치 완료')


### 5. 파이프라인 실행

`run_pipeline.sh`가 각 영상에 대해 아래 두 단계를 순서대로 실행합니다.  
이미 완료된 단계는 자동으로 건너뜁니다.

---

#### Stage 1 — 트랙 수집 (`collect_tracks.ipynb`)

YOLO v8n으로 사람을 탐지하고 BotSort 트래커로 각 인물에 `track_id`를 부여하여  
프레임별 bbox를 수집합니다.

| 항목 | 내용 |
|------|------|
| 입력 | `data/{영상}.avi` |
| 처리 | YOLO v8n 탐지 → BotSort 추적 → 프레임별 `track_id` + `bbox` 기록 |
| 생성 파일 | `tracks/{영상}.pkl` |
| 파일 내용 | `{'tracks': {track_id: [{'frame': int, 'bbox': [x1,y1,x2,y2]}]}, 'frames_processed': int, 'total_frames': int}` |

---

#### Stage 2 — Re-ID 임베딩 추출 (`reid_performance.ipynb`)

Stage 1의 bbox를 사용해 인물 영역을 크롭하고 OSNet으로 512차원 임베딩 벡터를 추출합니다.  
`FRAME_STEP`마다 샘플링하므로 Stage 1보다 빠릅니다.

| 항목 | 내용 |
|------|------|
| 입력 | `tracks/{영상}.pkl` + `data/{영상}.avi` |
| 처리 | bbox 크롭 → OSNet (`osnet_x0_25`) 추론 → 512-dim 임베딩 L2 정규화 |
| 생성 파일 | `results/{영상}.pkl` |
| 파일 내용 | `{'data_x025': {track_id: [{'frame': int, 'vector': ndarray(512,), 'bbox': list, 'image_path': str}]}, 'frames_processed': int, ...}` |
| 학습용 이미지 | `dataset/{영상}/track_{id:06d}/{frame:06d}.jpg` (`IMAGE_DIR` 지정 시) |

> **JPG 캐시 fast-path**: `dataset/` 이미지가 이미 존재하면 `data/*.avi`를 다시 읽지 않고  
> 저장된 JPG → ReID 추론만 수행합니다.  
> 모델 교체·파인튜닝 비교 시 `results/*.pkl`만 삭제하고 재실행하면 됩니다.

| 상황 | 동작 |
|------|------|
| **첫 실행** (`dataset/` 없음) | AVI → bbox 크롭 → ReID 추론 → JPG 저장 |
| **재실행** (`dataset/` 있음) | JPG 로드 → ReID 추론만 (`data/*.avi` 불필요) |

---

In [ ]:
import subprocess, shlex, sys, os, shutil

if not VIDEO_FILES:
    raise FileNotFoundError(f'영상 파일이 없습니다: {DATA_DIR}/*.avi')

# ── Stage 1 tracks를 /content/ SSD에서 처리 (Drive FUSE 체크포인트 쓰기 방지) ──
LOCAL_TRACKS  = '/content/ey_d_tracks'
DRIVE_TRACKS  = f'{DRIVE_ROOT}/tracks'
os.makedirs(LOCAL_TRACKS, exist_ok=True)
os.makedirs(DRIVE_TRACKS, exist_ok=True)

# 기존 tracks.pkl을 Drive → SSD로 복사 (재개 가능하도록)
_copied = 0
for _f in os.listdir(DRIVE_TRACKS):
    if _f.endswith('.pkl'):
        shutil.copy2(f'{DRIVE_TRACKS}/{_f}', f'{LOCAL_TRACKS}/{_f}')
        _copied += 1
if _copied:
    print(f'[스테이징] 기존 tracks.pkl {_copied}개 Drive → /content/ 복사 완료')

videos_str = ' '.join(shlex.quote(v) for v in VIDEO_FILES)

cmd = (
    f'COLAB=1 '
    f'DRIVE_ROOT={shlex.quote(DRIVE_ROOT)} '
    f'TRACKS_DIR={shlex.quote(LOCAL_TRACKS)} '
    f'IMAGE_DIR={shlex.quote(IMAGE_DIR)} '
    f'FRAME_STEP={FRAME_STEP} '
    f'MAX_FRAMES={MAX_FRAMES} '
    f'CLEAN={CLEAN} '
    f'bash {shlex.quote(SCRIPT)} {videos_str}'
)

print('실행 명령:')
print(cmd)
print()

process = subprocess.Popen(
    cmd, shell=True, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode != 0:
    print(f'[오류] 종료 코드: {process.returncode}')
else:
    print('[완료]')

# ── 완료 후 tracks.pkl을 SSD → Drive로 복사 ──────────────────────────────────
_saved = 0
for _f in os.listdir(LOCAL_TRACKS):
    if _f.endswith('.pkl'):
        shutil.copy2(f'{LOCAL_TRACKS}/{_f}', f'{DRIVE_TRACKS}/{_f}')
        _saved += 1
if _saved:
    print(f'[백업] tracks.pkl {_saved}개 /content/ → Drive 복사 완료')

### 6. 결과 확인


In [ ]:
import os, pickle

RESULTS_DIR = f'{DRIVE_ROOT}/results'
TRACKS_DIR  = f'{DRIVE_ROOT}/tracks'

print('── tracks.pkl ──────────────────────────────────')
for f in sorted(os.listdir(TRACKS_DIR)) if os.path.exists(TRACKS_DIR) else []:
    if not f.endswith('.pkl'):
        continue
    with open(f'{TRACKS_DIR}/{f}', 'rb') as fh:
        d = pickle.load(fh)
    total   = d.get('total_frames', '?')
    done    = d.get('frames_processed', '?')
    n_tracks = len(d.get('tracks', {}))
    print(f'  {f}: {done}/{total} 프레임  |  트랙 {n_tracks}개')

print()
print('── embeddings.pkl ──────────────────────────────')
for f in sorted(os.listdir(RESULTS_DIR)) if os.path.exists(RESULTS_DIR) else []:
    if not f.endswith('.pkl'):
        continue
    with open(f'{RESULTS_DIR}/{f}', 'rb') as fh:
        d = pickle.load(fh)
    done     = d.get('frames_processed', '완료')
    n_tracks = len(d.get('data_x025', {}))
    n_embeds = sum(len(v) for v in d.get('data_x025', {}).values())
    print(f'  {f}: {done} 프레임  |  트랙 {n_tracks}개  |  임베딩 {n_embeds}개')
